# 🕰 Wayback Machine CDX Fetcher
**Refactored from Google Apps Script → Python Colab**

| Step | Cell | Keterangan |
|------|------|------------|
| 1 | Install | Install dependencies |
| 2 | Config | Isi daftar domain & settings |
| 3 | Functions | Load semua fungsi (jangan diubah) |
| 4 | Run | Jalankan fetch |
| 5 | Download | Download hasil Excel |

## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install requests openpyxl tqdm -q
print('✅ Dependencies ready')

## ✏️ Step 2 — Input Domain & Config
> **Edit bagian ini sebelum run.**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ============================================================
# ⚙️ CONFIG
# ============================================================
OUTPUT_FILE      = "wayback_cdx_results.xlsx"
CDX_API          = "https://web.archive.org/cdx/search/cdx"
THRESHOLD_FULL   = 30_000
THRESHOLD_SAMPLE = 500_000
SAMPLE_LIMIT     = 10_000
SLEEP_BETWEEN    = 1.5

FORBIDDEN_KEYWORDS = {
    "judi": [
        "slot", "casino", "poker", "togel", "betting", "taruhan", "jackpot",
        "pragmatic", "gacor", "maxwin", "scatter", "rtp", "sbobet", "bet",
        "gambling", "lotere", "baccarat", "toto", "4d", "3d", "2d",
    ],
    "porno": [
        "porn", "xxx", "sex", "bokep", "nude", "naked", "adult", "hentai",
        "escort", "bugil", "telanjang", "ngentot", "memek", "kontol",
    ],
    "jasa_ilegal": ["hacker", "carding", "darkweb", "phishing", "skimming"],
}

# ============================================================
# ✏️ INPUT DOMAIN — ketik satu per baris, klik Simpan
# ============================================================
DOMAINS = []

txt_input = widgets.Textarea(
    placeholder="example.com\nanotherdomain.net\nketik satu domain per baris...",
    layout=widgets.Layout(width="500px", height="180px"),
)
btn_save  = widgets.Button(description="💾 Simpan Domain", button_style="primary")
lbl_info  = widgets.Label(value="")

def on_save(b):
    global DOMAINS
    lines  = txt_input.value.strip().splitlines()
    DOMAINS = [l.strip() for l in lines if l.strip()]
    lbl_info.value = f"✅ {len(DOMAINS)} domain tersimpan: {', '.join(DOMAINS[:3])}{'...' if len(DOMAINS) > 3 else ''}"

btn_save.on_click(on_save)

display(widgets.VBox([
    widgets.Label("📋 Daftar Domain (satu per baris):"),
    txt_input,
    btn_save,
    lbl_info,
]))

## ⚙️ Step 3 — Load Functions
> Jalankan cell ini sekali. Tidak perlu diubah.

In [ ]:
import re
import time
import requests
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from datetime import datetime
from tqdm.notebook import tqdm


def format_timestamp(ts: str) -> str:
    """'20231015123456' → '2023-10-15 12:34:56'"""
    if not ts or len(ts) < 8:
        return ts
    return f"{ts[0:4]}-{ts[4:6]}-{ts[6:8]} {ts[8:10]}:{ts[10:12]}:{ts[12:14]}"


def calc_domain_age(ts_start: str, ts_last: str) -> str:
    """Hitung umur domain dari dua timestamp. Returns: '18y 7m 3d'"""
    try:
        start = datetime.strptime(ts_start, "%Y-%m-%d %H:%M:%S")
        last  = datetime.strptime(ts_last,  "%Y-%m-%d %H:%M:%S")
    except ValueError:
        return ""
    diff_days = (last - start).days
    return f"{diff_days // 365}y {(diff_days % 365) // 30}m {diff_days % 30}d"


def count_cdx_snapshots(domain: str) -> int:
    """Phase 1: estimasi jumlah snapshot via showNumPages."""
    params = {
        "url": f"{domain}/*", "output": "json", "fl": "timestamp",
        "collapse": "digest", "limit": 1, "showNumPages": "true",
    }
    try:
        resp = requests.get(CDX_API, params=params, timeout=30)
        resp.raise_for_status()
        text = resp.text.strip()
        return int(text) if text.isdigit() else 0
    except Exception:
        return -1


def fetch_cdx(domain: str, fetch_limit: int) -> list:
    """
    Phase 2: fetch snapshot data.
    fetch_limit=0  → full mode (collapse=digest, limit 100k)
    fetch_limit>0  → sampled mode (collapse=urlkey)
    """
    is_sampled = fetch_limit > 0
    params = {
        "url":      f"{domain}/*",
        "output":   "json",
        "fl":       "timestamp,original,statuscode,mimetype,digest",
        "collapse": "urlkey" if is_sampled else "digest",
        "limit":    fetch_limit if is_sampled else 100_000,
    }
    resp = requests.get(CDX_API, params=params, timeout=60)
    resp.raise_for_status()
    raw = resp.json()
    return raw[1:] if raw and len(raw) > 1 else []


def extract_unique_slugs(urls: list) -> list:
    """Ekstrak & dedup slug dari list URL snapshot."""
    slug_set = set()
    for url in urls:
        parts = re.split(r"https?://web\.archive\.org/web/\d+/", url)
        for part in parts:
            if not part:
                continue
            m    = re.match(r"https?://[^/]+(?::\d+)?(/.*)?$", part)
            path = m.group(1) if m and m.group(1) else f"/{part}"
            for seg in path.split("/"):
                seg = re.sub(r"\?.*$", "", seg)
                seg = re.sub(r"\.[a-z]{2,4}$", "", seg)
                seg = re.sub(r"[^a-z0-9\-_]", "", seg.lower())
                if len(seg) > 2:
                    slug_set.add(seg)
    return list(slug_set)


def analyze_slugs(slugs: list) -> dict:
    """Cek slug terhadap FORBIDDEN_KEYWORDS."""
    flagged_count, flagged_examples = {}, {}
    for category, keywords in FORBIDDEN_KEYWORDS.items():
        for kw in keywords:
            pattern = re.compile(
                r"(?<![a-z0-9])" + re.escape(kw) + r"(?![a-z0-9])", re.IGNORECASE
            )
            for slug in slugs:
                if pattern.search(slug):
                    flagged_count[category] = flagged_count.get(category, 0) + 1
                    exs = flagged_examples.setdefault(category, [])
                    if len(exs) < 3 and slug not in exs:
                        exs.append(slug)

    is_flagged = bool(flagged_count)
    summary    = "flagged: " + ", ".join(f"{c} ({n})" for c, n in flagged_count.items()) if is_flagged else "clean"
    examples   = " | ".join(f"{c}: {', '.join(v)}" for c, v in flagged_examples.items()) if is_flagged else ""
    return {"flagged": is_flagged, "summary": summary, "examples": examples}


def make_header_style():
    return Font(bold=True, color="FFFFFF"), PatternFill("solid", fgColor="4285F4")


print('✅ Semua fungsi berhasil di-load')

## ▶️ Step 4 — Run Fetcher

In [ ]:
# ── Setup workbook ────────────────────────────────────────────
wb     = openpyxl.Workbook()
master = wb.active
master.title = "raw"

MASTER_HEADERS = [
    "domain", "status", "last_check",
    "snapshots_total", "snapshots_start", "snapshots_last", "domain_age",
    "slug_analysis", "flagged_examples",
]
MASTER_COL_WIDTHS = [25, 22, 20, 15, 20, 20, 15, 30, 45]

hdr_font, hdr_fill = make_header_style()
for col, h in enumerate(MASTER_HEADERS, 1):
    c = master.cell(row=1, column=col, value=h)
    c.font, c.fill = hdr_font, hdr_fill
    c.alignment = Alignment(horizontal="center")
for i, w in enumerate(MASTER_COL_WIDTHS, 1):
    master.column_dimensions[get_column_letter(i)].width = w
master.freeze_panes = "A2"

SNAP_HEADERS    = ["timestamp", "original", "statuscode", "mimetype", "digest", "wayback_url"]
SNAP_COL_WIDTHS = [18, 40, 12, 18, 22, 55]


def write_master_row(sheet, row_num, domain, **kwargs):
    vals = [
        domain,
        kwargs.get("status", ""),
        kwargs.get("last_check", datetime.now().strftime("%Y-%m-%d %H:%M:%S")),
        kwargs.get("snapshots_total", ""),
        kwargs.get("snapshots_start", ""),
        kwargs.get("snapshots_last", ""),
        kwargs.get("domain_age", ""),
        kwargs.get("slug_analysis", ""),
        kwargs.get("flagged_examples", ""),
    ]
    for col, val in enumerate(vals, 1):
        sheet.cell(row=row_num, column=col, value=val)


# ── Loop domain ───────────────────────────────────────────────
for row_num, domain in enumerate(tqdm(DOMAINS, desc="Processing domains"), start=2):
    domain = domain.strip()
    if not domain:
        continue

    print(f"\n▶ {domain}")

    try:
        # Phase 1 — estimasi
        total_pages = count_cdx_snapshots(domain)
        estimated   = total_pages * 3000
        print(f"  Estimasi snapshot: ~{estimated:,}")

        # Tentukan mode
        if estimated <= THRESHOLD_FULL:
            fetch_limit, fetch_mode = 0, "full"
        elif estimated <= THRESHOLD_SAMPLE:
            fetch_limit, fetch_mode = SAMPLE_LIMIT, "sampled"
        else:
            write_master_row(master, row_num, domain,
                status=f"too large: ~{estimated:,}", snapshots_total=estimated)
            print(f"  ⚠️ Skip — too large ({estimated:,})")
            wb.save(OUTPUT_FILE)
            continue

        print(f"  Mode: {fetch_mode} | limit: {fetch_limit or 'all'}")

        # Phase 2 — fetch
        snapshots = fetch_cdx(domain, fetch_limit)
        if not snapshots:
            write_master_row(master, row_num, domain, status="no snapshots")
            print("  ℹ️ Tidak ada snapshot")
            wb.save(OUTPUT_FILE)
            continue

        # Phase 3 — metadata
        timestamps = [s[0] for s in snapshots]
        ts_start   = format_timestamp(timestamps[0])
        ts_last    = format_timestamp(timestamps[-1])
        domain_age = calc_domain_age(ts_start, ts_last)

        # Phase 4 — slug analysis
        slugs       = extract_unique_slugs([s[1] for s in snapshots])
        slug_result = analyze_slugs(slugs)
        print(f"  ✅ {len(snapshots):,} snapshots | {len(slugs):,} slugs | {slug_result['summary']}")

        # Phase 5 — sheet per domain
        safe_name  = re.sub(r"[\\/*?:\[\]]", "_", domain)[:28]
        snap_sheet = wb.create_sheet(title=safe_name)

        for col, h in enumerate(SNAP_HEADERS, 1):
            c = snap_sheet.cell(row=1, column=col, value=h)
            c.font, c.fill = hdr_font, hdr_fill
            c.alignment = Alignment(horizontal="center")
        for i, w in enumerate(SNAP_COL_WIDTHS, 1):
            snap_sheet.column_dimensions[get_column_letter(i)].width = w
        snap_sheet.freeze_panes = "A2"

        for r, s in enumerate(snapshots, start=2):
            ts  = s[0]
            url = s[1]
            snap_sheet.cell(row=r, column=1, value=ts)
            snap_sheet.cell(row=r, column=2, value=url)
            snap_sheet.cell(row=r, column=3, value=s[2])
            snap_sheet.cell(row=r, column=4, value=s[3])
            snap_sheet.cell(row=r, column=5, value=s[4])
            snap_sheet.cell(row=r, column=6, value=f"https://web.archive.org/web/{ts}/{url}")

        # Phase 6 — tulis master row + row color
        row_fill = PatternFill("solid", fgColor="FCE8E6" if slug_result["flagged"] else "E6F4EA")
        write_master_row(master, row_num, domain,
            status="success" if fetch_mode == "full" else "success (sampled)",
            snapshots_total=len(snapshots),
            snapshots_start=ts_start,
            snapshots_last=ts_last,
            domain_age=domain_age,
            slug_analysis=slug_result["summary"],
            flagged_examples=slug_result["examples"],
        )
        for col in range(1, len(MASTER_HEADERS) + 1):
            master.cell(row=row_num, column=col).fill = row_fill

    except Exception as e:
        write_master_row(master, row_num, domain, status=f"error: {e}")
        print(f"  ❌ Error: {e}")

    wb.save(OUTPUT_FILE)
    time.sleep(SLEEP_BETWEEN)

wb.save(OUTPUT_FILE)
print(f"\n✅ Selesai! File: {OUTPUT_FILE}")

## 💾 Step 5 — Download Hasil Excel

In [ ]:
from google.colab import files
files.download(OUTPUT_FILE)
print(f'📥 Download dimulai: {OUTPUT_FILE}')